# 1장 — 큰 그림: 사전학습에서 비서까지 🗺️

웹 챕터에서 조련 파이프라인 3단계 지도를 봤죠. 이번엔 그 3단계를 **초소형 코드**로 세워 봅니다.

워밍업 노트북의 "이어쓰기 기계"가 1단계(사전학습) 역할을 그대로 맡고,
그 위에 2단계(SFT 흉내), 3단계(선호 학습 흉내)를 한 층씩 쌓아 —
**같은 요청 하나가 단계마다 어떻게 다른 응답이 되는지** 확인합니다.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch1')

## 1단계 — 사전학습: 이어쓰기 기계 (워밍업 재사용)

사전학습을 마친 베이스 모델의 본업은 **다음에 올 법한 단어 이어 붙이기**였죠.
워밍업에서 만든 초소형 기계를 그대로 다시 세웁니다 — 이게 파이프라인의 1층입니다.

In [ ]:
from collections import Counter, defaultdict

# 이 기계가 '인터넷'이라고 알고 있는 전부 (사전학습 데이터 — 라벨 없는 그냥 글)
CORPUS = [
    '회의록을 요약해 줘 라는 요청 메일을 받은 신입사원이 저지르기 쉬운 실수',
    '회의록을 잘 쓰는 법에 대한 글은 많지만 잘 요약하는 법은 드물다',
    '오늘 회의는 일정 확정과 담당자 배정으로 마무리되었다',
]

next_words = defaultdict(Counter)
for line in CORPUS:
    words = line.split()
    for a, b in zip(words, words[1:]):
        next_words[a][b] += 1

def base_model(prompt, length=10):
    """1단계(사전학습만): 프롬프트 뒤에 올 법한 단어를 이어 붙인다."""
    words = prompt.split()
    for _ in range(length):
        last = words[-1]
        if last not in next_words:
            break
        words.append(next_words[last].most_common(1)[0][0])
    return ' '.join(words)

REQUEST = '회의록을 요약해 줘'
print('요청  :', REQUEST)
print('1단계 :', base_model(REQUEST))  # 요청을 수행하지 않고 이어 쓴다

## 2단계 — SFT: 모범 답안 따라 하기

SFT 가 먹는 데이터는 **지시 → 모범 답안 쌍**(사람이 작성)이었죠.
초소형 버전: "이 지시에는 이 답변"이라는 시범 사례집을 주고, 요청이 들어오면
이어쓰기 대신 **사례집에서 배운 답변 행동**을 하게 합니다.

(진짜 SFT 는 사례집을 통째로 외우는 게 아니라 같은 다음-단어-예측 학습으로 행동을 배웁니다 — 3장에서 정확히)

In [ ]:
# SFT 데이터: 지시 → 모범 답안 (사람이 직접 쓴 시범, 수만 개 중 3개만 흉내)
SFT_EXAMPLES = {
    '회의록을 요약해 줘': '네, 요약해 드리겠습니다. 회의에서는 먼저 일정이 확정되었고, 이어서 담당자가 배정되었으며, 마지막으로 다음 회의 날짜가 정해졌습니다. 더 필요하신 것이 있다면 말씀해 주세요.',
    '번역해 줘': '네, 번역해 드리겠습니다. (원문을 함께 주시면 바로 번역할게요.)',
    '이메일 초안 써 줘': '네, 초안을 작성해 드리겠습니다. 받는 분과 용건을 알려 주시면 맞춰 쓸게요.',
}

def sft_model(prompt):
    """2단계(SFT 후): 요청에 '답변'으로 반응한다 — 아직 장황하지만."""
    for instruction, answer in SFT_EXAMPLES.items():
        if instruction in prompt:
            return answer
    return base_model(prompt)  # 배운 적 없는 요청이면 이어쓰기 본능이 새어 나온다

print('요청  :', REQUEST)
print('2단계 :', sft_model(REQUEST))

## 3단계 — 선호 학습: 사람이 고른 쪽으로 다듬기

선호 학습이 먹는 데이터는 **답변 A/B + 사람의 선택**이었죠.
초소형 버전: 후보 답변 여러 개에 사람들의 선호 점수를 매겨 두고, **가장 선호된 답**을 내게 합니다.
(점수를 '누가·어떻게' 매기는지가 4장 보상 모델, 그 점수로 '어떻게 배우는지'가 5~7장)

In [ ]:
# 같은 요청에 대한 후보 답변들 + 사람 선호 점수 (A/B 비교를 여러 번 모은 결과라고 상상)
PREFERENCES = {
    '회의록을 요약해 줘': [
        (sft_model('회의록을 요약해 줘'), 0.31),                      # 장황한 SFT 답
        ('요약: ① 일정 확정 ② 담당자 배정 ③ 다음 회의 날짜 결정', 0.62),  # 간결 — 사람들이 선호
        ('요약 불가. 회의록이 너무 길다.', 0.07),                      # 무성의 — 비선호
    ],
}

def chat_model(prompt):
    """3단계(선호 학습 후): 후보 중 사람이 가장 선호한 방향의 답을 낸다."""
    if prompt in PREFERENCES:
        return max(PREFERENCES[prompt], key=lambda pair: pair[1])[0]
    return sft_model(prompt)

print('요청  :', REQUEST)
print('3단계 :', chat_model(REQUEST))

In [ ]:
# 한눈에 비교 — 같은 요청, 세 단계의 응답
print('요청:', REQUEST)
print()
print('🦁 1단계 사전학습 :', base_model(REQUEST))
print('🏫 2단계 SFT      :', sft_model(REQUEST))
print('🐕 3단계 선호 학습:', chat_model(REQUEST))

# 검증 — 각 단계의 행동이 지도와 일치하는지
assert base_model(REQUEST).startswith(REQUEST), '1단계는 요청 뒤에 이어 쓸 뿐!'
assert not sft_model(REQUEST).startswith(REQUEST), '2단계부터는 이어쓰기가 아니라 답변!'
assert len(chat_model(REQUEST)) < len(sft_model(REQUEST)), '3단계는 사람이 선호한 간결한 쪽!'
print('\n✔ 확인: 이어쓰기(1) → 답변(2) → 다듬어진 답변(3) — 지도 그대로')

## 관찰 정리 + 🤖 AI 연결

- **지식은 사전학습이, 행동은 SFT가, 취향은 선호 학습이 만든다** — 코드로도 확인했습니다.
- 2단계에서 배운 적 없는 요청을 넣으면 이어쓰기 본능이 새어 나오는 것도 봤죠 —
  시범(데이터)이 부족한 곳에서 조련이 벗겨지는 현상은 9장(부작용)에서 다시 만납니다.
- 진짜 파이프라인과 다른 점: 진짜 모델은 사례집을 검색하는 게 아니라, **가중치가 바뀌어** 행동이 변합니다.
  그 "가중치를 바꾸는 수학"이 3장(크로스 엔트로피)부터의 내용이에요.

**다음 챕터**: 2장 — 조련 없이 프롬프트만으로 어디까지 버틸 수 있는지, 그 한계를 정확히 봅니다.

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch1')`.